In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms
import os

train_transforms = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

test_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

train_dir = os.path.join(path,'PlantVillage/train')
test_dir = os.path.join(path,'PlantVillage/test')

# Automatically handles everything if folders are named correctly
train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
test_dataset  = ImageFolder(root=test_dir,  transform=test_transforms)

train_dataloder = DataLoader(train_dataset, batch_size=128, shuffle= True, num_workers=2)
test_dataloder = DataLoader(test_dataset, batch_size=128, shuffle= False, num_workers=2)

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
# Create a function to visualize samples
def visualize_samples(dataset, num_samples=8, title="Dataset Samples"):
    """
    Visualize random samples from a dataset.

    Args:
        dataset: PyTorch Dataset object
        num_samples: Number of samples to display
        title: Title for the plot
    """
    # Select random indices
    indices = random.sample(range(len(dataset)), num_samples)

    # Calculate grid size
    cols = 4
    rows = (num_samples + cols - 1) // cols

    # Create the plot
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        # Get image and label
        image, label = dataset[idx]
        # Convert tensor to numpy for display
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        # Get class name
        class_name = dataset.classes[label]
        # Display image
        axes[i].imshow(image)
        axes[i].set_title(f"{class_name}\n(Label: {label})", fontsize=10)
        axes[i].axis('off')
    # Hide any unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
visualize_samples(train_dataset, num_samples=8, title="Training Dataset Samples")

In [ ]:
# Write your code here
import torch
import torch.nn as nn

class CustomModel(nn.Module):
    def __init__(self):
        """
        1️⃣ Define all layers in the model.
        """
        super(CustomModel, self).__init__()

        # Convolutional Layer + Activation + Pooling
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)


        #self.bn = nn.BatchNorm2d()
        # Fully Connected Layer
        self.fc = nn.Linear(16384, 3)  # Output 3 classes

        # Softmax Layer
        self.softmax = nn.Softmax(dim=1)  # Apply along the class dimension

    def forward(self, x):
        """
        2️⃣ Define the forward pass (how data flows through the model).
        """
        x = self.conv1(x)  # Convolution
        x = self.relu(x)  # Activation
        x = self.conv2(x) # Convolution
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling

        x = self.conv3(x) # Convolution
        x = self.relu(x)  # Activation
        x = self.conv4(x) # Convolution
        x = self.relu(x)  # Activation
        x = self.conv5(x) # Convolution
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling

        x = torch.flatten(x, start_dim=1)  # Flatten for FC layer
        x = self.fc(x)  # Fully connected layer
        x = self.softmax(x)  # Convert logits to probabilities
        return x  # Returns probability distribution


In [ ]:
# Write your code here
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

In [ ]:
# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

In [ ]:
# Write your code here
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []


for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_dataloder, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_dataloder, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
# Write your code here
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here
